# Single-Speaker Text-to-Speech: Fine-Tuning and Evaluation

This notebook reads LJSpeech text and audio, fine-tunes Glow-TTS, and compares losses, generated speech, and inference speed under two learning rates.
Both experiments use the same initial weights, data split, epoch count, batch size, and vocoder.

Before use, name this notebook **`3_A3.ipynb`** and save it in the **`Group3_A3`** folder on Google Drive.
The dataset folder is `Group3_A3/wav_dataset/`, containing `metadata.csv` and `wavs/`.
Use a fresh Colab GPU runtime and execute the cells from top to bottom. The first run installs dependencies and downloads the models; run evaluation and packaging after training finishes.
The default setup selects 2,000 usable records, uses an 80% / 10% / 10% split, and trains each experiment for 3 epochs with a batch size of 4.
If you change the sample count, epoch count, or batch size, rerun both experiments. These values are initial experimental settings.

The model was pretrained on LJSpeech, so the test set here is a **holdout set for this fine-tuning experiment**. It is not guaranteed to have been excluded from pretraining.
Validation loss selects the checkpoint. Test results are used only for the final comparison; speed metrics alone do not measure speech quality.

## 1. Install the Training and Audio Processing Environment

Install matching PyTorch extensions and the components for text-to-speech, English phoneme conversion, and audio analysis.
If the current session has already imported a different PyTorch version, restart the session after installation and run this module again.

In [1]:
# Install matching deep learning components and fixed versions of the TTS packages.
import importlib
import importlib.metadata
import subprocess
import sys

TORCH_VERSION = "2.8.0"
TORCHAUDIO_VERSION = "2.8.0"
TORCHVISION_VERSION = "0.23.0"
COQUI_VERSION = "0.28.0"
TRAINER_VERSION = "0.4.0"


def package_version(name):
    """Return the installed package version, or an empty string if the package is absent."""
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return ""


# Install PyTorch only when versions differ, preserving an existing CUDA build of the same version.
torch_was_loaded = "torch" in sys.modules
loaded_tts_mismatch = any(
    module in sys.modules and package_version(package) != version
    for module, package, version in [
        ("TTS", "coqui-tts", COQUI_VERSION),
        ("trainer", "coqui-tts-trainer", TRAINER_VERSION),
        ("coqpit", "coqpit-config", "0.2.5"),
    ]
)
torch_packages = {
    "torch": TORCH_VERSION,
    "torchaudio": TORCHAUDIO_VERSION,
    "torchvision": TORCHVISION_VERSION,
}
replace_torch = any(package_version(k).split("+")[0] != v for k, v in torch_packages.items())
if replace_torch:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet",
         *[f"{k}=={v}" for k, v in torch_packages.items()],
         "--index-url", "https://download.pytorch.org/whl/cu128"],
        check=True,
    )

# Install components for training, audio decoding, tables, plotting, and notebook packaging.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     f"coqui-tts=={COQUI_VERSION}", f"coqui-tts-trainer=={TRAINER_VERSION}",
     "coqpit-config==0.2.5", "gruut==2.4.0", "soundfile==0.13.1",
     "pandas>=2.2,<3", "matplotlib>=3.8,<4", "tqdm>=4.66", "nbformat>=5.10"],
    check=True,
)
importlib.invalidate_caches()
if (replace_torch and torch_was_loaded) or loaded_tts_mismatch:
    raise RuntimeError("Dependencies are installed. Restart the Colab session, then run the modules in order from Module 1.")

# Check core imports in a separate process so dependency errors appear before further execution.
subprocess.run(
    [sys.executable, "-c",
     "import torch, torchaudio, gruut; from TTS.api import TTS; "
     "from trainer import Trainer; print('Core dependencies imported successfully. PyTorch:', torch.__version__)"],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-c', "import torch, torchaudio, gruut; from TTS.api import TTS; from trainer import Trainer; print('Core dependencies imported successfully. PyTorch:', torch.__version__)"], returncode=0)

## 2. Mount Google Drive and Set Project Paths

Keep the original audio on Google Drive and cache the selected audio on the Colab local disk.
Save models, experiment logs, result tables, and generated audio in the project folder. Rebuild the local cache if the runtime is recreated.

In [2]:
# Mount Google Drive and define the input, cache, and output directories.
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
GROUP_NUMBER = "3"
PROJECT_DIR = Path("/content/drive/MyDrive/Group3_A3")
DATA_DIR = PROJECT_DIR / "wav_dataset"
METADATA_PATH = DATA_DIR / "metadata.csv"
WAV_DIR = DATA_DIR / "wavs"
OUTPUT_DIR = PROJECT_DIR / "wav_examples"
RESULTS_DIR = PROJECT_DIR / "results"
SPLITS_DIR = PROJECT_DIR / "splits"
MODEL_CACHE_DIR = PROJECT_DIR / "model_cache"
RUNS_DIR = PROJECT_DIR / "runs"
LOCAL_CACHE_BASE = Path("/content/Group3_A3_cache")

# Create output directories and check the data inputs without modifying the original files.
for directory in [OUTPUT_DIR, RESULTS_DIR, SPLITS_DIR, MODEL_CACHE_DIR, RUNS_DIR, LOCAL_CACHE_BASE]:
    directory.mkdir(parents=True, exist_ok=True)
assert METADATA_PATH.is_file(), f"Transcript file not found: {METADATA_PATH}"
assert WAV_DIR.is_dir(), f"Audio directory not found: {WAV_DIR}"
print("Project directory:", PROJECT_DIR)
print("Dataset directory:", DATA_DIR)

Mounted at /content/drive
Project directory: /content/drive/MyDrive/Group3_A3
Dataset directory: /content/drive/MyDrive/Group3_A3/wav_dataset


## 3. Set Experimental Parameters and Random Seeds

expA and expB differ only in learning rate. Both use Adam, a fixed learning rate, the same pretrained weights, and the same training budget.
The vocoder generates audio and is not fine-tuned. If GPU memory is insufficient, reduce the shared batch size here and rerun both experiments.

In [3]:
# Import numerical, audio, plotting, and training tools, and define the experimental parameters.
import csv
import gc
import hashlib
import json
import math
import os
import platform
import random
import shutil
import time
import zipfile
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
from IPython.display import Audio, display
from tqdm.auto import tqdm

from TTS.api import TTS
from TTS.config import load_config
from TTS.tts.configs.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.glow_tts import GlowTTS
from TTS.utils.manage import ModelManager
from trainer import Trainer, TrainerArgs

SEED = 42
PIPELINE_VERSION = "1.0.0"
SUBSET_SIZE = 2000
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.8, 0.1, 0.1
EPOCHS = 3
BATCH_SIZE = 4
NUM_WORKERS = 0
MIN_DURATION_SEC, MAX_DURATION_SEC = 0.5, 10.5
MAX_TEXT_CHARS = 400
BENCHMARK_SENTENCES = 10
TIMING_REPEATS = 3
WARMUP_RUNS = 2
SKIP_COMPLETED_EXPERIMENTS = True
MODEL_NAME = "tts_models/en/ljspeech/glow-tts"
EXPERIMENTS = [{"name": "expA", "lr": 1e-5}, {"name": "expB", "lr": 5e-5}]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MIXED_PRECISION = DEVICE == "cuda"


def seed_everything(seed):
    """Set the random states for Python, NumPy, PyTorch, and CUDA."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False


# Check parameter ranges and fix random states before training.
assert math.isclose(TRAIN_RATIO + VAL_RATIO + TEST_RATIO, 1.0)
assert SUBSET_SIZE >= 100 and BATCH_SIZE >= 1 and EPOCHS >= 1
assert BENCHMARK_SENTENCES >= 10 and TIMING_REPEATS >= 1
seed_everything(SEED)
print("Device:", DEVICE)
print("Training settings:", {"subset": SUBSET_SIZE, "epochs": EPOCHS, "batch_size": BATCH_SIZE})
display(pd.DataFrame(EXPERIMENTS))

Device: cuda
Training settings: {'subset': 2000, 'epochs': 3, 'batch_size': 4}


,name,lr
0,expA,0.00001
1,expB,0.00005


## 4. Define File Verification and Result-Saving Functions

Identify data and model files using SHA-256, and save reusable experiment records as JSON and CSV files.

In [4]:
# Define hashing and atomic writes to reduce incomplete records after interruptions.
def sha256_file(path):
    """Compute a file SHA-256 in chunks without loading a large model file into memory at once."""
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def content_hash(value):
    """Generate a stable identifier for a serializable configuration."""
    raw = json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def save_json(path, value):
    """Write a temporary file before replacing the destination JSON file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def read_json(path):
    """Read a UTF-8 JSON file."""
    return json.loads(Path(path).read_text(encoding="utf-8"))


def save_dataframe(frame, path):
    """Save a table as UTF-8 while preserving column names and values."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=False, encoding="utf-8-sig")
    temporary.replace(path)


def release_memory():
    """Collect unreferenced objects and release unused CUDA cache memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Download Pretrained Models and Record the Environment

Download Glow-TTS and its default vocoder while preserving the original text frontend, vocabulary, and audio configuration.
Record model file checksums, software versions, and hardware information.

In [5]:
# Download the complete model resources to persistent storage, including auxiliary files referenced by configurations.
manager = ModelManager(output_prefix=str(MODEL_CACHE_DIR), progress_bar=True)
PRETRAINED_PATH, PRETRAINED_CONFIG_PATH, model_item = manager.download_model(MODEL_NAME)
VOCODER_NAME = model_item["default_vocoder"]
assert VOCODER_NAME, "The selected acoustic model has no default vocoder."
VOCODER_PATH, VOCODER_CONFIG_PATH, vocoder_item = manager.download_model(VOCODER_NAME)

# Read the sample rate from the pretrained configuration while preserving the model architecture and text representation.
base_config = load_config(str(PRETRAINED_CONFIG_PATH))
SAMPLE_RATE = int(base_config.audio.sample_rate)
assert base_config.model.lower() == "glow_tts"
assert SAMPLE_RATE == 22050, f"The model sample rate differs from the dataset setting: {SAMPLE_RATE}"
assets = {
    "model_name": MODEL_NAME,
    "model_path": str(Path(PRETRAINED_PATH).resolve()),
    "model_config_path": str(Path(PRETRAINED_CONFIG_PATH).resolve()),
    "vocoder_name": VOCODER_NAME,
    "vocoder_path": str(Path(VOCODER_PATH).resolve()),
    "vocoder_config_path": str(Path(VOCODER_CONFIG_PATH).resolve()),
    "model_sha256": sha256_file(PRETRAINED_PATH),
    "model_config_sha256": sha256_file(PRETRAINED_CONFIG_PATH),
    "vocoder_sha256": sha256_file(VOCODER_PATH),
    "vocoder_config_sha256": sha256_file(VOCODER_CONFIG_PATH),
    "model_license": model_item.get("license"),
    "vocoder_license": vocoder_item.get("license"),
    "sample_rate": SAMPLE_RATE,
    "phonemizer": base_config.phonemizer,
    "text_cleaner": base_config.text_cleaner,
}

# Record the current software, hardware, and complete package list.
environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "gpu_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if DEVICE == "cuda" else None,
    "torch": torch.__version__,
    "torchaudio": torchaudio.__version__,
    "cuda_runtime": torch.version.cuda,
    "coqui_tts": package_version("coqui-tts"),
    "trainer": package_version("coqui-tts-trainer"),
    "numpy": np.__version__,
    "soundfile": sf.__version__,
    "librosa": package_version("librosa"),
    "gruut": package_version("gruut"),
    "coqpit_config": package_version("coqpit-config"),
    "mixed_precision": MIXED_PRECISION,
}
save_json(RESULTS_DIR / "model_assets.json", assets)
save_json(RESULTS_DIR / "environment.json", environment)
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RESULTS_DIR / "requirements_freeze.txt").write_text(freeze, encoding="utf-8")
display(pd.DataFrame([environment]))
print("Sample rate:", SAMPLE_RATE, "; phonemizer:", base_config.phonemizer)

100%|██████████| 344M/344M [00:22<00:00, 15.3MiB/s]
100%|██████████| 82.8M/82.8M [00:04<00:00, 16.9MiB/s]
/usr/local/lib/python3.13/dist-packages/coqpit/coqpit.py:875: UserWarning: Type mismatch in MultibandMelganConfig
Failed to deserialize field: steps_to_start_discriminator (<class 'int'>) = True
Value `True` does not match field type `<class 'int'>`
Replaced it with field's default value: 200000
  self.deserialize(data)


,python,platform,device,gpu,gpu_vram_gb,torch,torchaudio,cuda_runtime,coqui_tts,trainer,numpy,soundfile,librosa,gruut,coqpit_config,mixed_precision
0,"3.13.16 (main, Oct 1 2026, 15:40:24) [GCC 13....",Linux-6.6.159+-x86_64-with-glibc2.39,cuda,Tesla T4,14.56,2.8.0+cu128,2.8.0+cu128,12.8,0.28.0,0.4.0,2.1.3,0.13.1,0.11.0,2.4.0,0.2.5,True


Sample rate: 22050 ; phonemizer: gruut


## 6. Read Transcripts and Check Audio Filenames

The three columns in metadata.csv contain the audio ID, original transcript, and normalized transcript.
Match WAV files by audio ID and check for duplicate IDs, empty fields, missing files, and extra files.

In [6]:
# Read pipe-delimited transcripts while preserving quotation marks and Unicode characters.
with METADATA_PATH.open("r", encoding="utf-8-sig", newline="") as file:
    metadata_rows = list(csv.reader(file, delimiter="|", quoting=csv.QUOTE_NONE))
assert metadata_rows and all(len(row) == 3 for row in metadata_rows), "metadata.csv must contain three columns with no header row."
metadata = pd.DataFrame(metadata_rows, columns=["id", "raw_text", "text"])
for column in metadata.columns:
    metadata[column] = metadata[column].str.strip()
assert not metadata.eq("").any().any(), "The metadata contains empty fields."
assert metadata["id"].is_unique, "The metadata contains duplicate audio IDs."
assert metadata["id"].str.fullmatch(r"LJ\d{3}-\d{4}").all(), "An audio ID has an unexpected format."

# List the audio directory once and compare sets instead of checking remote files individually.
wav_ids = {path.stem for path in WAV_DIR.iterdir() if path.suffix.lower() == ".wav"}
metadata_ids = set(metadata["id"])
missing_ids = sorted(metadata_ids - wav_ids)
extra_ids = sorted(wav_ids - metadata_ids)
inventory = {
    "metadata_rows": len(metadata), "wav_files": len(wav_ids),
    "missing_ids": missing_ids, "extra_ids": extra_ids,
    "normalized_text_changes": int((metadata["raw_text"] != metadata["text"]).sum()),
    "duplicate_text_rows": int(metadata.duplicated("text").sum()),
    "metadata_sha256": sha256_file(METADATA_PATH),
}
save_json(RESULTS_DIR / "data_inventory.json", inventory)
assert not missing_ids, f"Missing {len(missing_ids)} audio files; examples: {missing_ids[:5]}"
print({key: value for key, value in inventory.items() if key not in ["missing_ids", "extra_ids"]})
print("Number of extra audio files:", len(extra_ids))
display(metadata.head())

{'metadata_rows': 13100, 'wav_files': 13100, 'normalized_text_changes': 1505, 'duplicate_text_rows': 26, 'metadata_sha256': '852bce5c79e5184ba5bf2ea98aad4bcf49a5d787def5fd1f04fa2ab8aa6dc535'}
Number of extra audio files: 0


,id,raw_text,text
0,LJ001-0001,"Printing, in the only sense with which we are ...","Printing, in the only sense with which we are ..."
1,LJ001-0002,in being comparatively modern.,in being comparatively modern.
2,LJ001-0003,For although the Chinese took impressions from...,For although the Chinese took impressions from...
3,LJ001-0004,"produced the block books, which were the immed...","produced the block books, which were the immed..."
4,LJ001-0005,the invention of movable metal letters in the ...,the invention of movable metal letters in the ...


## 7. Cache a Subset and Validate Audio Content

Copy candidate audio files to local storage in a fixed random order. Check their sample rate, channel count, duration, numerical validity, and signal level.
Remove exact duplicate normalized transcripts to avoid placing the same text in different splits. Save reasons for rejected samples rather than discarding them silently.

In [7]:
# Create a separate cache directory based on the metadata and filtering parameters.
data_spec = {
    "metadata_sha256": inventory["metadata_sha256"], "seed": SEED,
    "subset_size": SUBSET_SIZE, "ratios": [TRAIN_RATIO, VAL_RATIO, TEST_RATIO],
    "min_duration_sec": MIN_DURATION_SEC, "max_duration_sec": MAX_DURATION_SEC,
    "max_text_chars": MAX_TEXT_CHARS, "sample_rate": SAMPLE_RATE,
    "deduplicate_normalized_text": True,
}
DATA_ID = content_hash(data_spec)[:12]
LOCAL_DATA_DIR = LOCAL_CACHE_BASE / DATA_ID
LOCAL_WAV_DIR = LOCAL_DATA_DIR / "wavs"
LOCAL_WAV_DIR.mkdir(parents=True, exist_ok=True)


def validate_audio(path, expected_sr, min_duration, max_duration):
    """Decode the complete audio file and return its format, duration, amplitude statistics, and checksum."""
    info = sf.info(path)
    if info.samplerate != expected_sr:
        raise ValueError(f"sample_rate={info.samplerate}")
    if info.channels != 1:
        raise ValueError(f"channels={info.channels}")
    audio, sr = sf.read(path, dtype="float32", always_2d=True)
    if not audio.size or not np.isfinite(audio).all():
        raise ValueError("empty_or_nonfinite_audio")
    duration = len(audio) / sr
    if not min_duration <= duration <= max_duration:
        raise ValueError(f"duration_sec={duration:.4f}")
    rms = float(np.sqrt(np.mean(np.square(audio, dtype=np.float64))))
    if rms < 1e-5:
        raise ValueError("near_silent_audio")
    return {"duration_sec": duration, "sample_rate": sr, "channels": info.channels,
            "subtype": info.subtype, "frames": len(audio), "rms": rms,
            "peak": float(np.max(np.abs(audio))),
            "near_full_scale_fraction": float(np.mean(np.abs(audio) >= 0.999)),
            "audio_sha256": sha256_file(path)}


# Remove duplicate texts and fix the candidate order, selecting replacements for rejected audio samples.
candidates = metadata.sort_values("id").drop_duplicates("text", keep="first").reset_index(drop=True)
order = np.random.default_rng(SEED).permutation(len(candidates))
selected_records, rejected_records = [], []
for index in tqdm(order, desc="Caching and validating audio"):
    row = candidates.iloc[int(index)].to_dict()
    if not 1 <= len(row["text"]) <= MAX_TEXT_CHARS:
        rejected_records.append({"id": row["id"], "reason": "text_length"})
        continue
    source = WAV_DIR / f"{row['id']}.wav"
    destination = LOCAL_WAV_DIR / source.name
    try:
        # Rename the temporary copy only after copying completes, so interrupted copies are not treated as valid cache files.
        if not destination.is_file():
            temporary = destination.with_suffix(".wav.partial")
            shutil.copyfile(source, temporary)
            temporary.replace(destination)
        stats = validate_audio(destination, SAMPLE_RATE, MIN_DURATION_SEC, MAX_DURATION_SEC)
        selected_records.append({**row, **stats, "audio_path": str(destination)})
    except (OSError, RuntimeError, ValueError) as error:
        rejected_records.append({"id": row["id"], "reason": f"{type(error).__name__}: {error}"})
    if len(selected_records) == SUBSET_SIZE:
        break

# Save the selected samples and rejection records, then check that the target sample count has been reached.
selected = pd.DataFrame(selected_records)
rejected = pd.DataFrame(rejected_records, columns=["id", "reason"])
save_dataframe(rejected, RESULTS_DIR / f"rejected_{DATA_ID}.csv")
assert len(selected) == SUBSET_SIZE, f"Found only {len(selected)} usable records; the target is {SUBSET_SIZE}."
print("Usable subset:", len(selected), "; rejected records:", len(rejected))
display(selected[["id", "duration_sec", "sample_rate", "subtype", "peak"]].head())

Caching and validating audio:   0%|          | 0/13074 [00:00<?, ?it/s]

Usable subset: 2000 ; rejected records: 0


,id,duration_sec,sample_rate,subtype,peak
0,LJ002-0058,7.611655,22050,PCM_16,0.518616
1,LJ041-0203,6.810567,22050,PCM_16,0.553528
2,LJ044-0078,4.697551,22050,PCM_16,0.706421
3,LJ004-0025,6.113968,22050,PCM_16,0.336243
4,LJ050-0074,6.903447,22050,PCM_16,0.579102


## 8. Save Fixed Training, Validation, and Test Splits

Create disjoint ID sets and save the transcripts, audio checksums, and split records.
On subsequent runs with the same configuration, verify the saved split. Both experiments read the same explicit split files without creating new splits automatically.

In [8]:
# Split the subset in its fixed order and check split sizes and intersections.
n_train = int(len(selected) * TRAIN_RATIO)
n_val = int(len(selected) * VAL_RATIO)
assert n_train >= BATCH_SIZE and n_val > 0 and len(selected) - n_train - n_val >= BENCHMARK_SENTENCES
selected["split"] = (["train"] * n_train + ["validation"] * n_val
                     + ["test"] * (len(selected) - n_train - n_val))
train_df = selected[selected["split"] == "train"].copy()
val_df = selected[selected["split"] == "validation"].copy()
test_df = selected[selected["split"] == "test"].copy()
assert not (set(train_df.id) & set(val_df.id) or set(train_df.id) & set(test_df.id) or set(val_df.id) & set(test_df.id))
assert selected["id"].is_unique and selected["text"].is_unique

# Save split records without temporary Colab paths and reject conflicting content on subsequent runs.
persistent_columns = [column for column in selected.columns if column != "audio_path"]
split_identity = [{"id": row.id, "split": row.split, "audio_sha256": row.audio_sha256}
                  for row in selected.itertuples(index=False)]
split_info = {"data_spec": data_spec, "data_id": DATA_ID,
              "split_sha256": content_hash(split_identity), "samples": split_identity}
split_path = SPLITS_DIR / f"split_{DATA_ID}.json"
if split_path.exists():
    assert read_json(split_path) == split_info, "The saved split differs from the current data. Check the audio files or modified parameters."
else:
    save_json(split_path, split_info)
save_dataframe(selected[persistent_columns], SPLITS_DIR / f"samples_{DATA_ID}.csv")

# Write three explicit metadata files, retaining the normalized transcript in the third column.
for name, frame in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    path = LOCAL_DATA_DIR / f"metadata_{name}.csv"
    lines = [f"{row.id}|{row.raw_text}|{row.text}\n" for row in frame.itertuples(index=False)]
    path.write_text("".join(lines), encoding="utf-8")
display(selected.groupby("split").agg(samples=("id", "size"), hours=("duration_sec", lambda x: x.sum() / 3600)))

,samples,hours
split,,
test,200,0.377101
train,1600,2.932001
validation,200,0.343480


## 9. Build Shared Training Configurations and Check Text Encoding

Build both training configurations from the pretrained configuration, preserving the model architecture, phoneme vocabulary, and audio processing settings.
Disable data-dependent initialization to preserve the learned normalization parameters. Use a fixed optimizer and learning rate without a learning rate scheduler.

In [9]:
# Collect the settings that identify the experiment and store different configurations separately.
protocol = {
    "pipeline_version": PIPELINE_VERSION,
    "data_id": DATA_ID, "split_sha256": split_info["split_sha256"],
    "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "optimizer": "Adam", "experiments": EXPERIMENTS, "scheduler": None,
    "model_sha256": assets["model_sha256"], "vocoder_sha256": assets["vocoder_sha256"],
    "model_config_sha256": assets["model_config_sha256"],
    "vocoder_config_sha256": assets["vocoder_config_sha256"],
    "frontend_versions": {key: environment[key] for key in ["numpy", "soundfile", "librosa", "gruut", "coqpit_config"]},
    "coqui_version": package_version("coqui-tts"), "trainer_version": package_version("coqui-tts-trainer"),
    "torch": torch.__version__, "gpu": environment["gpu"], "mixed_precision": MIXED_PRECISION,
    "num_workers": NUM_WORKERS, "benchmark_sentences": BENCHMARK_SENTENCES,
    "timing_repeats": TIMING_REPEATS, "warmup_runs": WARMUP_RUNS,
    "inference_noise_scale": 0.0, "length_scale": 1.0,
    "latency_scope": "text_frontend_and_acoustic_model_and_vocoder_excluding_load_and_file_save",
}
RUN_ID = content_hash(protocol)[:12]
RUN_DIR = RUNS_DIR / f"glowtts_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
save_json(RUN_DIR / "protocol.json", protocol)
save_json(RUN_DIR / "environment.json", environment)
save_json(RUN_DIR / "model_assets.json", assets)
(RUN_DIR / "requirements_freeze.txt").write_text(freeze, encoding="utf-8")


def make_training_config(experiment):
    """Preserve the pretrained architecture and set the fixed data inputs and current experiment learning rate."""
    config = load_config(str(PRETRAINED_CONFIG_PATH))
    config.datasets = [BaseDatasetConfig(
        formatter="ljspeech", dataset_name="ljspeech",
        path=str(LOCAL_DATA_DIR), meta_file_train="metadata_train.csv",
        meta_file_val="metadata_validation.csv", language="en-us",
    )]
    config.output_path = str(RUN_DIR / experiment["name"])
    config.run_name = experiment["name"]
    config.run_description = f"Glow-TTS fine-tuning, learning rate {experiment['lr']}"
    config.epochs = EPOCHS
    config.batch_size = BATCH_SIZE
    config.eval_batch_size = BATCH_SIZE
    config.num_loader_workers = NUM_WORKERS
    config.num_eval_loader_workers = NUM_WORKERS
    config.precompute_num_workers = 0
    config.training_seed = SEED
    config.cudnn_deterministic = True
    config.cudnn_benchmark = False
    config.allow_tf32 = False
    config.mixed_precision = MIXED_PRECISION
    config.precision = "fp16"
    config.optimizer = "Adam"
    config.optimizer_params = {"betas": [0.9, 0.999], "weight_decay": 0.0}
    config.lr = float(experiment["lr"])
    config.lr_scheduler = None
    config.lr_scheduler_params = {}
    config.grad_clip = 1.0
    config.data_dep_init_steps = 0
    config.shuffle = True
    config.drop_last = False
    config.batch_group_size = 0
    config.min_text_len = 1
    config.max_text_len = MAX_TEXT_CHARS
    config.min_audio_len = 1
    config.max_audio_len = int(MAX_DURATION_SEC * SAMPLE_RATE) + 1
    config.phoneme_cache_path = str(LOCAL_DATA_DIR / "phoneme_cache")
    config.compute_input_seq_cache = True
    config.run_eval = True
    config.test_delay_epochs = 10**9
    config.test_sentences = []
    config.print_step = 50
    config.plot_step = 100000
    config.print_eval = False
    config.save_step = 200
    config.save_n_checkpoints = 1
    config.save_all_best = False
    config.save_best_after = 0
    config.save_on_interrupt = True
    config.target_loss = "loss"
    config.inference_noise_scale = 0.0
    config.length_scale = 1.0
    return config


# Load the shared data lists and strictly restore the weights once to verify vocabulary and architecture compatibility.
preview_config = make_training_config(EXPERIMENTS[0])
preview_model = GlowTTS(preview_config)
preview_model.load_checkpoint(preview_model.config, str(PRETRAINED_PATH), eval=False, strict=True)
preview_train, preview_val = load_tts_samples(preview_model.config, eval_split=True)
assert len(preview_train) == len(train_df) and len(preview_val) == len(val_df)
assert {Path(row["audio_file"]).stem for row in preview_train} == set(train_df.id)
assert {Path(row["audio_file"]).stem for row in preview_val} == set(val_df.id)
token_ids = preview_model.tokenizer.text_to_ids(train_df.iloc[0]["text"])
assert len(token_ids) > 0
print("Training / validation:", len(preview_train), len(preview_val))
print("Example transcript:", train_df.iloc[0]["text"])
print("Encoded length:", len(token_ids), "; model parameters:", sum(p.numel() for p in preview_model.parameters()))
del preview_model, preview_config, preview_train, preview_val
release_memory()

Training / validation: 1600 200
Example transcript: The chapel yard led to the chapel, and on the staircase were two rooms frequently set apart for the king's witnesses,
Encoded length: 104 ; model parameters: 28609873


## 10. Verify Speech Generation with the Pretrained Model

Use an independent sentence to check that the model, text frontend, and vocoder can synthesize speech together.
This audio checks the pipeline and is excluded from the speed statistics for the two fine-tuned experiments.

In [10]:
# Define shared functions for loading synthesis models and saving WAV files.
def load_synthesizer(checkpoint, config_path):
    """Load the acoustic model and shared pretrained vocoder with fixed synthesis parameters."""
    synthesizer = TTS(
        model_path=str(checkpoint), config_path=str(config_path),
        vocoder_path=str(VOCODER_PATH), vocoder_config_path=str(VOCODER_CONFIG_PATH),
        progress_bar=False,
    ).to(DEVICE)
    synthesizer.synthesizer.tts_model.inference_noise_scale = 0.0
    synthesizer.synthesizer.tts_model.length_scale = 1.0
    return synthesizer


def save_generated_audio(path, waveform, sample_rate):
    """Validate generated audio, limit its peak when necessary, and save a playable 16-bit WAV file."""
    waveform = np.asarray(waveform, dtype=np.float32).reshape(-1)
    if not waveform.size or not np.isfinite(waveform).all():
        raise ValueError("The generated audio is empty or contains invalid numerical values.")
    peak = float(np.max(np.abs(waveform)))
    scale = 0.99 / peak if peak > 1.0 else 1.0
    sf.write(str(path), waveform * scale, sample_rate, subtype="PCM_16")
    return {"duration_sec": len(waveform) / sample_rate,
            "peak_before_scaling": peak, "save_scale": scale,
            "wav_sha256": sha256_file(path)}


# Generate and play one audio clip to check the pretrained synthesis pipeline.
seed_everything(SEED)
baseline_synthesizer = load_synthesizer(PRETRAINED_PATH, PRETRAINED_CONFIG_PATH)
baseline_waveform = baseline_synthesizer.tts(text="This is a short test of the text to speech system.")
baseline_sr = int(baseline_synthesizer.synthesizer.output_sample_rate)
baseline_path = RUN_DIR / "pretrained_smoke.wav"
save_generated_audio(baseline_path, baseline_waveform, baseline_sr)
display(Audio(filename=str(baseline_path)))
del baseline_synthesizer, baseline_waveform
release_memory()

## 11. Define Independent Fine-Tuning and Epoch Logging

Create a new model for each experiment and strictly load the same initial weights. Neither experiment inherits the other experiment's parameters or optimizer state.
Save the best model according to validation loss and record losses, training time, and peak GPU memory usage.
Completed experiments can be reused. Interrupted experiments retain their checkpoints, but calling this function again restarts training from the initial weights.

In [11]:
# Define the training entry point for one experiment and save its configuration, logs, and completion status.
def train_experiment(experiment):
    """Fine-tune one model independently and return its best checkpoint and training statistics."""
    experiment_dir = RUN_DIR / experiment["name"]
    experiment_dir.mkdir(parents=True, exist_ok=True)
    completion_path = experiment_dir / "completed.json"
    if SKIP_COMPLETED_EXPERIMENTS and completion_path.exists():
        previous = read_json(completion_path)
        checkpoint = Path(previous["checkpoint_path"])
        if (previous["protocol_sha256"] == content_hash(protocol) and checkpoint.is_file()
                and sha256_file(checkpoint) == previous["checkpoint_sha256"]):
            print(f"Reusing completed experiment: {experiment['name']}")
            return previous

    # Reset random states and verify the initial checkpoint so both experiments start independently.
    assert sha256_file(PRETRAINED_PATH) == assets["model_sha256"]
    seed_everything(SEED)
    release_memory()
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    config = make_training_config(experiment)
    model = GlowTTS(config)
    model.load_checkpoint(model.config, str(PRETRAINED_PATH), eval=False, strict=True)
    model.run_data_dep_init = False
    train_samples, val_samples = load_tts_samples(model.config, eval_split=True)
    assert {Path(row["audio_file"]).stem for row in train_samples} == set(train_df.id)
    assert {Path(row["audio_file"]).stem for row in val_samples} == set(val_df.id)
    model.config.save_json(str(experiment_dir / "requested_config.json"))
    history_records = []
    epoch_clock = {"start": None}

    # Collect training metrics, validation metrics, and timing at epoch boundaries.
    def epoch_start(trainer):
        """Record the start time of the current training epoch."""
        epoch_clock["start"] = time.perf_counter()

    def epoch_end(trainer):
        """Write the average losses and elapsed time for the current epoch to CSV."""
        row = {"experiment": experiment["name"], "epoch": int(trainer.epochs_done) + 1,
               "step": int(trainer.total_steps_done),
               "epoch_seconds": time.perf_counter() - epoch_clock["start"]}
        for prefix, averages in [("train", trainer.keep_avg_train), ("validation", trainer.keep_avg_eval)]:
            if averages is None:
                continue
            for key, value in averages.avg_values.items():
                numeric = float(value)
                if not math.isfinite(numeric):
                    raise FloatingPointError(f"{prefix}/{key} contains a non-finite value.")
                row[f"{prefix}_{key}"] = numeric
        history_records.append(row)
        save_dataframe(pd.DataFrame(history_records), experiment_dir / "history.csv")

    # Start the trainer with explicit training and validation lists to avoid creating new data splits.
    trainer = None
    start = time.perf_counter()
    save_json(experiment_dir / "status.json", {"state": "running", "experiment": experiment,
              "protocol_sha256": content_hash(protocol), "started_utc": datetime.now(timezone.utc).isoformat()})
    try:
        trainer = Trainer(
            TrainerArgs(), model.config, str(experiment_dir), model=model,
            train_samples=train_samples, eval_samples=val_samples,
            parse_command_line_args=False,
            callbacks={"on_epoch_start": epoch_start, "on_epoch_end": epoch_end},
        )
        trainer.fit()
        assert len(history_records) == EPOCHS, "Training has not completed all epochs."
        checkpoint = Path(trainer.output_path) / "best_model.pth"
        trained_config = Path(trainer.output_path) / "config.json"
        assert checkpoint.is_file() and trained_config.is_file(), "The best model checkpoint or its configuration was not found."
        best_loss = min(float(row["validation_avg_loss"]) for row in history_records)
        result = {
            "experiment": experiment["name"], "lr": float(experiment["lr"]),
            "epochs": EPOCHS, "batch_size": BATCH_SIZE, "seed": SEED,
            "protocol_sha256": content_hash(protocol),
            "initial_checkpoint_sha256": assets["model_sha256"],
            "split_sha256": split_info["split_sha256"],
            "checkpoint_path": str(checkpoint), "config_path": str(trained_config),
            "checkpoint_sha256": sha256_file(checkpoint), "best_validation_loss": best_loss,
            "training_seconds": time.perf_counter() - start,
            "peak_gpu_memory_gb": torch.cuda.max_memory_allocated() / 1024**3 if DEVICE == "cuda" else None,
            "training_device": DEVICE, "training_gpu": environment["gpu"],
            "discarded_phoneme_symbols": list(model.tokenizer.not_found_characters),
            "history_path": str(experiment_dir / "history.csv"),
        }
        save_json(completion_path, result)
        save_json(experiment_dir / "status.json", {"state": "completed", **result})
        return result
    except BaseException as error:
        save_json(experiment_dir / "status.json", {"state": "interrupted_or_failed",
                  "error": f"{type(error).__name__}: {error}", "protocol_sha256": content_hash(protocol)})
        raise
    finally:
        # Release the current model so the next experiment does not inherit its GPU memory allocation.
        trainer = None
        model = None
        release_memory()

## 12. Fine-Tune expA

Run the first experiment independently from the pretrained weights with a learning rate of `1e-5`.

In [12]:
# Train the first model and display its best validation loss and training time.
result_a = train_experiment(EXPERIMENTS[0])
display(pd.DataFrame([result_a])[["experiment", "lr", "best_validation_loss", "training_seconds"]])

 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: True
 | > Precision: fp16
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 2
 | > Num. of Torch Threads: 1
 | > Torch seed: 42
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: True
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=/content/drive/MyDrive/Group3_A3/runs/glowtts_7f100a4d1220/expA/expA-October-28-2026_12+42PM-0000000

 > Model has 28609873 parameters

 > EPOCH: 0/2
 --> /content/drive/MyDrive/Group3_A3/runs/glowtts_7f100a4d1220/expA/expA-October-28-2026_12+42PM-0000000
100%|██████████| 1600/1600 [00:00<00:00, 9795.91it/s] 

 > TRAINING (2026-10-28 12:44:09) 

   --> TIME: 2026-10-28 12:44:27 -- STEP: 0/399 -- GLOBAL_STEP: 0
     | > current_lr: 1e-05  (1e-05)
     | > loss: -0.7273572683334351  (-0.7273572683334351)
     | > log_mle: -0.8991885185241699  (-0.8991885185241699)
     | > loss_dur: 0.17183123528957367  (0.1718312352895

,experiment,lr,best_validation_loss,training_seconds
0,expA,0.00001,-0.791456,605.129504


## 13. Fine-Tune expB and Verify Shared Conditions

Reload the same initial weights and use a learning rate of `5e-5`, keeping the data, optimizer, and training budget unchanged.

In [13]:
# Train the second model and verify the initial checkpoint and data split identifiers.
result_b = train_experiment(EXPERIMENTS[1])
experiment_results = {"expA": result_a, "expB": result_b}
assert result_a["initial_checkpoint_sha256"] == result_b["initial_checkpoint_sha256"]
assert result_a["split_sha256"] == result_b["split_sha256"]
assert result_a["epochs"] == result_b["epochs"] and result_a["batch_size"] == result_b["batch_size"]
save_json(RUN_DIR / "training_results.json", experiment_results)
display(pd.DataFrame(experiment_results.values())[["experiment", "lr", "epochs", "batch_size", "best_validation_loss", "training_seconds"]])

 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: True
 | > Precision: fp16
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 2
 | > Num. of Torch Threads: 1
 | > Torch seed: 42
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: True
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=/content/drive/MyDrive/Group3_A3/runs/glowtts_7f100a4d1220/expB/expB-October-28-2026_12+52PM-0000000

 > Model has 28609873 parameters

 > EPOCH: 0/2
 --> /content/drive/MyDrive/Group3_A3/runs/glowtts_7f100a4d1220/expB/expB-October-28-2026_12+52PM-0000000
100%|██████████| 1600/1600 [00:00<00:00, 10453.27it/s]

 > TRAINING (2026-10-28 12:52:41) 

   --> TIME: 2026-10-28 12:52:42 -- STEP: 0/399 -- GLOBAL_STEP: 0
     | > current_lr: 5e-05  (5e-05)
     | > loss: -0.7679283022880554  (-0.7679283022880554)
     | > log_mle: -0.9133630990982056  (-0.9133630990982056)
     | > loss_dur: 0.14543481171131134  (0.1454348117113

,experiment,lr,epochs,batch_size,best_validation_loss,training_seconds
0,expA,0.00001,3,4,-0.791456,605.129504
1,expB,0.00005,3,4,-0.794818,475.820864


## 14. Compare Training and Validation Losses

Inspect total loss across epochs for substantial fluctuations or diverging training and validation trends.
A decrease in loss indicates improvement in the model objective; it does not directly establish an improvement in perceived speech quality.

In [14]:
# Combine epoch logs and plot training and validation curves for both experiments.
history_frames = [pd.read_csv(result["history_path"]) for result in experiment_results.values()]
history = pd.concat(history_frames, ignore_index=True)
save_dataframe(history, RUN_DIR / "training_history.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for axis, column, title in zip(axes, ["train_avg_loss", "validation_avg_loss"], ["Training loss", "Validation loss"]):
    for name, frame in history.groupby("experiment", sort=False):
        axis.plot(frame["epoch"], frame[column], marker="o", label=name)
    axis.set(xlabel="Epoch", ylabel="Glow-TTS loss", title=title)
    axis.grid(alpha=0.25)
    axis.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "loss_curves.png", dpi=160, bbox_inches="tight")
plt.show()
display(history)

,experiment,epoch,step,epoch_seconds,train_avg_loader_time,train_avg_step_time,train_avg_current_lr,train_avg_loss,train_avg_log_mle,train_avg_loss_dur,train_avg_amp_scaler,train_avg_grad_norm,validation_avg_loader_time,validation_avg_loss,validation_avg_log_mle,validation_avg_loss_dur
0,expA,1,400,278.390505,0.056307,0.280567,0.00001,-0.727381,-0.902798,0.175417,16671.438596,0.651590,0.184915,-0.786332,-0.934946,0.148614
1,expA,2,800,153.511165,0.057976,0.242894,0.00001,-0.733969,-0.904978,0.171009,16384.000000,0.650158,0.044844,-0.789683,-0.934313,0.144630
2,expA,3,1200,158.307221,0.063404,0.247304,0.00001,-0.736857,-0.906188,0.169332,16384.000000,0.651446,0.045777,-0.791456,-0.935708,0.144253
3,expB,1,400,159.233448,0.061638,0.242345,0.00005,-0.730673,-0.901524,0.170851,17041.002506,0.706203,0.047262,-0.787990,-0.931638,0.143648
4,expB,2,800,155.558516,0.060472,0.239438,0.00005,-0.743340,-0.905435,0.162095,16384.000000,0.728892,0.045502,-0.791997,-0.931795,0.139798
5,expB,3,1200,158.747518,0.064943,0.245352,0.00005,-0.750993,-0.907953,0.156960,16384.000000,0.716012,0.045106,-0.794818,-0.931843,0.137025


## 15. Compute Model Losses on the Holdout Test Set

For each experiment, load the best model selected using the validation set and compute losses on the complete holdout test set.
Aggregate batch losses using sample-count weights. Use the same batch size and full-precision forward passes for both models.
Test loss is not used to select checkpoints and is not a speech naturalness score.

In [15]:
# Define test data loading and batch-wise loss computation.
def move_batch_to_device(value, device):
    """Recursively move batch tensors while preserving strings and None values."""
    if torch.is_tensor(value):
        return value.to(device)
    if isinstance(value, dict):
        return {key: move_batch_to_device(item, device) for key, item in value.items()}
    if isinstance(value, list):
        return [move_batch_to_device(item, device) for item in value]
    if isinstance(value, tuple):
        return tuple(move_batch_to_device(item, device) for item in value)
    return value


def evaluate_test_loss(result):
    """Compute sample-count-weighted mean batch losses on the fixed holdout set."""
    seed_everything(SEED)
    config = load_config(result["config_path"])
    config.datasets[0].path = str(LOCAL_DATA_DIR)
    config.datasets[0].meta_file_train = "metadata_test.csv"
    config.datasets[0].meta_file_val = ""
    config.eval_batch_size = BATCH_SIZE
    config.num_eval_loader_workers = 0
    config.data_dep_init_steps = 0
    model = GlowTTS(config)
    model.load_checkpoint(model.config, result["checkpoint_path"], eval=False, strict=True)
    model.to(DEVICE).eval()
    model.run_data_dep_init = False
    samples, _ = load_tts_samples(model.config, eval_split=False)
    assert {Path(row["audio_file"]).stem for row in samples} == set(test_df.id)
    loader = model.get_data_loader(model.config, is_eval=True, samples=samples, verbose=False)
    assert len(loader.dataset) == len(test_df), "The test data loader filtered out samples."
    criterion = model.get_criterion().to(DEVICE)
    sums, sample_count = {}, 0
    try:
        with torch.inference_mode():
            for raw_batch in tqdm(loader, desc=f"{result['experiment']} test loss"):
                batch = model.format_batch(raw_batch)
                batch = model.format_batch_on_device(move_batch_to_device(batch, DEVICE))
                _, losses = model.eval_step(batch, criterion)
                count = int(batch["text_input"].shape[0])
                for key, value in losses.items():
                    scalar = float(value.detach().cpu()) if torch.is_tensor(value) else float(value)
                    if not math.isfinite(scalar):
                        raise FloatingPointError(f"Test metric {key} contains a non-finite value.")
                    sums[key] = sums.get(key, 0.0) + count * scalar
                sample_count += count
        assert sample_count == len(test_df)
        return {"experiment": result["experiment"], "test_samples": sample_count,
                **{f"test_{key}": value / sample_count for key, value in sums.items()}}
    finally:
        del model, criterion, loader
        release_memory()


# Evaluate each best checkpoint and save the model loss table.
test_loss_table = pd.DataFrame([evaluate_test_loss(result) for result in experiment_results.values()])
save_dataframe(test_loss_table, RUN_DIR / "test_losses.csv")
display(test_loss_table)

100%|██████████| 200/200 [00:00<00:00, 13215.61it/s]


expA test loss:   0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 200/200 [00:00<00:00, 7628.43it/s]


expB test loss:   0%|          | 0/50 [00:00<?, ?it/s]

,experiment,test_samples,test_loss,test_log_mle,test_loss_dur
0,expA,200,-0.789224,-0.940078,0.150854
1,expB,200,-0.793461,-0.936813,0.143352


## 16. Fix the Speech Generation and Timing Protocol

Select the same 10 distinct sentences from the holdout test set. Warm up each model, then synthesize each sentence 3 times.
Latency includes text processing, the acoustic model, and the vocoder. It excludes model loading, file writing, and warm-up.
When using a GPU, synchronize before and after timing. The real-time factor is `RTF = synthesis time / generated audio duration`; lower values indicate faster generation.
Both experiments use fixed sampling noise and speaking-rate settings. Changing the learning rate does not necessarily improve inference speed.

In [16]:
# Save shared test sentences and timing parameters so both experiments use identical inputs.
benchmark_df = test_df.head(BENCHMARK_SENTENCES).copy().reset_index(drop=True)
assert len(benchmark_df) == BENCHMARK_SENTENCES and benchmark_df["text"].is_unique
save_dataframe(benchmark_df[["id", "text", "duration_sec"]], RUN_DIR / "benchmark_sentences.csv")
timing_protocol = {
    "warmup_runs": WARMUP_RUNS, "repeats_per_sentence": TIMING_REPEATS,
    "sentences_per_experiment": BENCHMARK_SENTENCES, "device": DEVICE,
    "gpu": environment["gpu"], "includes": ["text_frontend", "acoustic_model", "vocoder"],
    "excludes": ["model_loading", "warmup", "file_writing"],
    "inference_noise_scale": 0.0, "length_scale": 1.0,
    "rtf_definition": "latency_sec / generated_duration_sec",
}
save_json(RUN_DIR / "timing_protocol.json", timing_protocol)


def synchronize_device():
    """Wait for CUDA work to finish so timing covers the actual computation."""
    if DEVICE == "cuda":
        torch.cuda.synchronize()


def benchmark_experiment(result):
    """Repeatedly synthesize the same sentences, record latency, and save the first output for each sentence."""
    seed_everything(SEED)
    synthesizer = load_synthesizer(result["checkpoint_path"], result["config_path"])
    sample_rate = int(synthesizer.synthesizer.output_sample_rate)
    assert sample_rate == SAMPLE_RATE
    trials, audio_records = [], []
    try:
        # Warm up the acoustic model and vocoder, excluding warm-up outputs from the statistics.
        for _ in range(WARMUP_RUNS):
            _ = synthesizer.tts(text="This short sentence warms up the speech synthesis system.")
        synchronize_device()
        for index, row in tqdm(benchmark_df.iterrows(), total=len(benchmark_df), desc=f"{result['experiment']} synthesis timing"):
            for repeat in range(TIMING_REPEATS):
                seed_everything(SEED + int(index))
                synchronize_device()
                start = time.perf_counter()
                waveform = synthesizer.tts(text=row["text"])
                synchronize_device()
                latency = time.perf_counter() - start
                waveform = np.asarray(waveform, dtype=np.float32).reshape(-1)
                if not waveform.size or not np.isfinite(waveform).all():
                    raise ValueError(f"{result['experiment']} / {row['id']} produced invalid audio.")
                duration = len(waveform) / sample_rate
                trials.append({
                    "experiment": result["experiment"], "id": row["id"], "repeat": repeat + 1,
                    "text": row["text"], "latency_sec": latency,
                    "generated_duration_sec": duration, "rtf": latency / duration,
                })
                # Write files after timing ends and save only one audio file per sentence.
                if repeat == 0:
                    path = OUTPUT_DIR / f"{result['experiment']}_{int(index) + 1:02d}.wav"
                    stats = save_generated_audio(path, waveform, sample_rate)
                    audio_records.append({
                        "experiment": result["experiment"], "id": row["id"], "text": row["text"],
                        "wav_path": str(path), "sample_rate": sample_rate, **stats,
                    })
        # Record out-of-vocabulary phoneme symbols encountered by the text frontend.
        save_json(RUN_DIR / f"{result['experiment']}_frontend_audit.json", {
            "phonemizer": synthesizer.synthesizer.tts_model.config.phonemizer,
            "discarded_phoneme_symbols": list(synthesizer.synthesizer.tts_model.tokenizer.not_found_characters),
        })
    finally:
        del synthesizer
        release_memory()
    return pd.DataFrame(trials), pd.DataFrame(audio_records)

## 17. Generate Audio and Summarize Speed Metrics

Save every timing trial and the audio manifest. First take the median across repeated runs for each sentence, then summarize by experiment.
Keep the mean RTF and the ratio of total synthesis time to total audio duration separately because they use different aggregation methods.

In [17]:
# Run the same timing procedure for both best models and save the raw records.
trial_frames, audio_frames = [], []
for name in ["expA", "expB"]:
    trial_frame, audio_frame = benchmark_experiment(experiment_results[name])
    trial_frames.append(trial_frame)
    audio_frames.append(audio_frame)
    save_dataframe(trial_frame, RUN_DIR / f"{name}_timing_trials.csv")
    save_dataframe(audio_frame, RUN_DIR / f"{name}_audio_manifest.csv")
timing_trials = pd.concat(trial_frames, ignore_index=True)
audio_manifest = pd.concat(audio_frames, ignore_index=True)
save_dataframe(timing_trials, RUN_DIR / "timing_trials.csv")
save_dataframe(audio_manifest, RUN_DIR / "audio_manifest.csv")

# Summarize repeated measurements by sentence and verify identical input IDs across experiments.
per_sentence = timing_trials.groupby(["experiment", "id"], as_index=False).agg(
    latency_sec=("latency_sec", "median"),
    generated_duration_sec=("generated_duration_sec", "median"),
    rtf=("rtf", "median"),
)
assert set(per_sentence[per_sentence.experiment == "expA"].id) == set(per_sentence[per_sentence.experiment == "expB"].id)
speed_records = []
for name, frame in per_sentence.groupby("experiment", sort=False):
    speed_records.append({
        "experiment": name, "benchmark_sentences": len(frame),
        "mean_latency_sec": float(frame.latency_sec.mean()),
        "median_latency_sec": float(frame.latency_sec.median()),
        "latency_std_sec": float(frame.latency_sec.std(ddof=1)),
        "mean_rtf": float(frame.rtf.mean()), "rtf_std": float(frame.rtf.std(ddof=1)),
        "aggregate_rtf": float(frame.latency_sec.sum() / frame.generated_duration_sec.sum()),
    })
speed_table = pd.DataFrame(speed_records)
training_table = pd.DataFrame(experiment_results.values())[
    ["experiment", "lr", "epochs", "batch_size", "best_validation_loss", "training_seconds", "peak_gpu_memory_gb"]
]
results_table = training_table.merge(test_loss_table, on="experiment").merge(speed_table, on="experiment")
save_dataframe(per_sentence, RUN_DIR / "per_sentence_timing.csv")
save_dataframe(results_table, RUN_DIR / "experiment_results.csv")
save_dataframe(results_table, RESULTS_DIR / "latest_experiment_results.csv")
display(results_table)

expA synthesis timing:   0%|          | 0/10 [00:00<?, ?it/s]

expB synthesis timing:   0%|          | 0/10 [00:00<?, ?it/s]

,experiment,lr,epochs,batch_size,best_validation_loss,training_seconds,peak_gpu_memory_gb,test_samples,test_loss,test_log_mle,test_loss_dur,benchmark_sentences,mean_latency_sec,median_latency_sec,latency_std_sec,mean_rtf,rtf_std,aggregate_rtf
0,expA,0.00001,3,4,-0.791456,605.129504,0.676019,200,-0.789224,-0.940078,0.150854,10,0.099633,0.096390,0.031695,0.015024,0.003635,0.014375
1,expB,0.00005,3,4,-0.794818,475.820864,0.674274,200,-0.793461,-0.936813,0.143352,10,0.097615,0.100533,0.032384,0.014300,0.001750,0.013981


## 18. Plot Speed Comparisons and Verify Audio Outputs

Compare per-sentence latency and RTF. Error bars show the standard deviation across sentences, not confidence intervals for speech quality.
Check generated audio durations, sample rates, and checksums to avoid mixing old files into the current results.

In [18]:
# Plot comparisons of mean latency and RTF.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, metric, error, ylabel in [
    (axes[0], "mean_latency_sec", "latency_std_sec", "Latency (seconds)"),
    (axes[1], "mean_rtf", "rtf_std", "Real-time factor"),
]:
    axis.bar(speed_table["experiment"], speed_table[metric], yerr=speed_table[error], capsize=5)
    axis.set_ylabel(ylabel)
    axis.grid(axis="y", alpha=0.25)
    axis.set_axisbelow(True)
fig.tight_layout()
fig.savefig(RUN_DIR / "speed_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

# Check output counts, audio formats, and actual file contents for each experiment.
assert len(audio_manifest) == len(EXPERIMENTS) * BENCHMARK_SENTENCES
for row in audio_manifest.itertuples(index=False):
    path = Path(row.wav_path)
    assert path.is_file() and sha256_file(path) == row.wav_sha256
    info = sf.info(path)
    assert info.frames > 0 and info.channels == 1 and info.samplerate == SAMPLE_RATE
display(audio_manifest[["experiment", "id", "duration_sec", "peak_before_scaling", "save_scale"]])

,experiment,id,duration_sec,peak_before_scaling,save_scale
0,expA,LJ012-0293,9.346757,0.418419,1.0
1,expA,LJ033-0112,2.961270,0.418544,1.0
2,expA,LJ011-0246,6.026304,0.399300,1.0
3,expA,LJ014-0225,5.701224,0.354746,1.0
4,expA,LJ003-0116,10.241451,0.432830,1.0
5,expA,LJ041-0185,10.298776,0.465610,1.0
6,expA,LJ001-0177,5.120726,0.431391,1.0
7,expA,LJ015-0187,7.605261,0.353222,1.0
8,expA,LJ030-0151,4.168707,0.354370,1.0
9,expA,LJ030-0060,7.837460,0.342025,1.0


## 19. Listen to Matched Texts and Record Errors

Compare intelligibility, naturalness, pauses, mispronunciations, omissions, repetitions, and noise sentence by sentence.
The listening review table starts empty and should be completed after listening. Individual subjective ratings are not labelled as formal MOS results.
Keep the corresponding real recording as a reference. Focus on content and pronunciation; the generated and reference audio do not need to have identical durations.

In [19]:
# Play the reference recordings and both generated versions of the first three test sentences.
for sample_id in benchmark_df["id"].head(3):
    text = benchmark_df.loc[benchmark_df.id == sample_id, "text"].iloc[0]
    print(f"\n{sample_id}: {text}")
    print("Reference")
    display(Audio(filename=str(LOCAL_WAV_DIR / f"{sample_id}.wav")))
    for name in ["expA", "expB"]:
        path = audio_manifest.loc[(audio_manifest.id == sample_id) & (audio_manifest.experiment == name), "wav_path"].iloc[0]
        print(name)
        display(Audio(filename=str(path)))

# Create a listening review table only if absent, preserving existing human-entered records on subsequent runs.
listening_path = RUN_DIR / "listening_review.csv"
if not listening_path.exists():
    listening = audio_manifest[["experiment", "id", "text", "wav_path"]].copy()
    for column in ["reviewer", "intelligibility_1_to_5", "naturalness_1_to_5", "mispronunciation",
                   "omission", "repetition", "pause_or_prosody", "noise_or_artifact", "notes"]:
        listening[column] = ""
    save_dataframe(listening, listening_path)
print("Listening review:", listening_path)

Output hidden; open in https://colab.research.google.com to view.

## 20. Synthesize Arbitrary English Text

Edit `CUSTOM_TEXT` to synthesize a new sentence using the best model from either experiment.
Record this demonstration audio separately from the fixed timing set and exclude it from the earlier speed statistics.

In [20]:
# Synthesize any non-empty English text and save it as a separately named WAV file.
def synthesize_custom_text(text, experiment="expA"):
    """Synthesize custom text using the best model from the specified experiment."""
    text = text.strip()
    if not text:
        raise ValueError("Enter non-empty text.")
    if experiment not in experiment_results:
        raise ValueError("experiment must be expA or expB.")
    seed_everything(SEED)
    result = experiment_results[experiment]
    synthesizer = load_synthesizer(result["checkpoint_path"], result["config_path"])
    try:
        waveform = synthesizer.tts(text=text)
        sr = int(synthesizer.synthesizer.output_sample_rate)
        path = OUTPUT_DIR / f"{experiment}_custom.wav"
        stats = save_generated_audio(path, waveform, sr)
        display(Audio(filename=str(path)))
        return {"experiment": experiment, "id": "custom", "text": text,
                "wav_path": str(path), "sample_rate": sr, **stats}
    finally:
        del synthesizer
        release_memory()


# Demonstrate the complete synthesis pipeline for both models using the same new text.
CUSTOM_TEXT = "Our speech system turns written sentences into clear spoken English."
custom_audio = pd.DataFrame([synthesize_custom_text(CUSTOM_TEXT, name) for name in ["expA", "expB"]])
save_dataframe(custom_audio, RUN_DIR / "custom_audio_manifest.csv")

## 21. Organize Outputs and Create the Submission Archive

Before packaging, save the current notebook so that training logs, figures, and result tables are stored in `3_A3.ipynb`.
Include the notebook and WAV files generated by the current experiments in the ZIP file. Exclude the original dataset, caches, and model checkpoints.
Keep the complete experiment records in the project's `runs/`, `splits/`, `model_cache/`, and `results/` folders.

In [21]:
# Verify the current audio and notebook, then create a ZIP archive with a fixed structure.
notebook_path = PROJECT_DIR / f"{GROUP_NUMBER}_A3.ipynb"
assert notebook_path.is_file(), f"Save the current notebook to this path first: {notebook_path}"
required_columns = ["experiment", "mean_latency_sec", "mean_rtf", "best_validation_loss"]
assert set(required_columns).issubset(results_table.columns)
assert set(results_table.experiment) == {"expA", "expB"}
assert np.isfinite(results_table[["mean_latency_sec", "mean_rtf", "best_validation_loss"]].to_numpy()).all()
assert audio_manifest.groupby("experiment")["id"].nunique().min() >= 10

# Verify that audio matches the current manifest and package only the listed files.
pack_audio = pd.concat([audio_manifest, custom_audio], ignore_index=True)
assert pack_audio["wav_path"].is_unique
for row in pack_audio.itertuples(index=False):
    assert sha256_file(row.wav_path) == row.wav_sha256, f"The audio file has changed: {row.wav_path}"
submission_dir = PROJECT_DIR / "submission"
submission_dir.mkdir(exist_ok=True)
zip_path = submission_dir / f"{GROUP_NUMBER}_A3.zip"
temporary_zip = zip_path.with_suffix(".zip.tmp")
with zipfile.ZipFile(temporary_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(notebook_path, arcname=f"{GROUP_NUMBER}_A3.ipynb")
    for path in pack_audio["wav_path"]:
        archive.write(path, arcname=f"wav_examples/{Path(path).name}")
temporary_zip.replace(zip_path)

# Recheck the archive contents and save the final file manifest.
with zipfile.ZipFile(zip_path) as archive:
    members = archive.namelist()
    assert archive.testzip() is None, "ZIP archive verification failed."
assert f"{GROUP_NUMBER}_A3.ipynb" in members
assert len([name for name in members if name.endswith(".wav")]) == len(pack_audio)
save_json(RUN_DIR / "submission_manifest.json", {"zip_path": str(zip_path), "members": members,
          "zip_sha256": sha256_file(zip_path), "run_id": RUN_ID})
print("ZIP archive:", zip_path)
print("Number of audio files:", len(pack_audio))
print("Complete experiment records:", RUN_DIR)

ZIP archive: /content/drive/MyDrive/Group3_A3/submission/3_A3.zip
Number of audio files: 22
Complete experiment records: /content/drive/MyDrive/Group3_A3/runs/glowtts_7f100a4d1220


## 22. Technical Documentation

- [LJ Speech Dataset and File Format](https://keithito.com/LJ-Speech-Dataset/)
- [Glow-TTS: Generative Flow for Text-to-Speech via Monotonic Alignment Search](https://arxiv.org/abs/2005.11129)
- [Coqui TTS 0.28.0](https://github.com/idiap/coqui-ai-TTS/tree/v0.28.0)
- [Training and Fine-Tuning Interfaces](https://coqui-tts.readthedocs.io/en/latest/training/finetuning.html)
- [Speech Synthesis Interfaces](https://coqui-tts.readthedocs.io/en/latest/inference.html)
- [Multi-band MelGAN](https://arxiv.org/abs/2005.05106)
